In [120]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [121]:
df = pd.read_csv("preprocessed_housing_ordered.csv")
df.sample(5)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
18640,-1.203053,0.631180,0.028646,0.020736,0.226908,0.200032,0.315077,-0.418824,0.077511,0,0,0,0,1
13677,1.157836,-0.735924,-0.845393,1.087861,1.361912,1.344469,1.212230,-0.456091,-0.869689,0,1,0,0,0
10923,0.843383,-0.890426,0.346478,-0.158035,0.136299,1.595256,0.233993,-0.351079,-0.375724,1,0,0,0,0
4455,0.698635,-0.721879,1.299975,-0.599920,-0.633883,-0.314788,-0.563767,-0.248961,-0.459784,1,0,0,0,0
13944,1.177801,-0.632923,-1.004309,0.316396,0.076687,-0.793403,-0.804403,0.985079,-0.340193,0,1,0,0,0


In [122]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20640 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   <1H OCEAN           20640 non-null  int64  
 10  INLAND              20640 non-null  int64  
 11  ISLAND              20640 non-null  int64  
 12  NEAR BAY            20640 non-null  int64  
 13  NEAR OCEAN          20640 non-null  int64  
dtypes: float64(9), int64(5)
memory usage: 2.2 MB


In [123]:
X = df.drop(columns="median_house_value").head(1000)
y = df["median_house_value"].head(1000)

In [124]:
print(X.shape)
print(y.shape)
print()

(1000, 13)
(1000,)



In [125]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)
X_train = X_train.drop(columns=["NEAR OCEAN"])
X_test = X_test.drop(columns=["NEAR OCEAN"])

# Sklearn's LinearRegression class implementation

In [126]:
reg = LinearRegression()
reg.fit(X_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [127]:
print(reg.coef_)
print(reg.intercept_)

[-0.18221792  3.50640384  0.01099591 -0.01544988  0.07555414 -0.22938627
  0.25382925  0.64545404  0.11001802  0.03955055  0.         -0.14956857]
-3.5523772818913755


In [128]:
y_pred = reg.predict(X_test)
r2_score(y_test,y_pred)

0.5770812944294635

# Manual implementation of Multiple Linear Regression

In [132]:
class ManualLinearRegression:
        
    def __init__(self):
        self.coef_ = None
        self.intercept_ = None
        
    def fit(self,X_train,y_train):
        X_train = X_train.to_numpy()
        y_train = y_train.to_numpy()
        
        X_train = np.insert(X_train,0,1,axis=1)
        
        # calcuate the coeffs
        # betas = np.linalg.inv(np.dot(X_train.T,X_train)).dot(X_train.T).dot(y_train)
        betas = np.linalg.pinv(X_train).dot(y_train)
        self.intercept_ = betas[0]
        self.coef_ = betas[1:]
    
    def predict(self,X_test):
        y_pred = np.dot(X_test,self.coef_) + self.intercept_
        return y_pred

In [133]:
LR = ManualLinearRegression()

In [134]:
LR.fit(X_train , y_train)

In [135]:
print(reg.coef_)
print(reg.intercept_)

[-0.18221792  3.50640384  0.01099591 -0.01544988  0.07555414 -0.22938627
  0.25382925  0.64545404  0.11001802  0.03955055  0.         -0.14956857]
-3.5523772818913755


In [136]:
y_pred = reg.predict(X_test)
r2_score(y_test,y_pred)

0.5770812944294635

# Manual implementation of Batch Gradient Descent Regression 

In [56]:
class ManualBatchGDR:

    def __init__(self,learning_Rate=0.1,epochs=1000):
        self.learning_Rate = learning_Rate
        self.epochs = epochs
        self.intercept_ = None
        self.coef_ = None

    
    def fit(self , X_train , y_train):
        self.intercept_ = 0
        self.coef_ = np.zeros(X_train.shape[1])

        for i in range(self.epochs):
            y_hat =  np.dot(X_train , self.coef_) + self.intercept_
            interceptDer = -2 * np.mean(y_train - y_hat)
            self.intercept_ = self.intercept_ - (self.learning_Rate * interceptDer)
    
            
            coefDer = -2 * np.dot((y_train - y_hat) , X_train) / X_train.shape[0]
            self.coef_ = self.coef_ - (self.learning_Rate * coefDer)
        
    def predict(self , X_test):
        return np.dot(X_test , self.coef_) + self.intercept_

In [57]:
model = ManualBatchGDR()

In [58]:
model.fit(X_train , y_train)

In [59]:
print(model.coef_)
print(model.intercept_)

[-0.09560526  0.34303725  0.08992519  0.02847993  0.13440524 -0.34813794
  0.26611552  0.64034056 -0.07634719 -0.02085773  0.         -0.2106989
  0.        ]
-0.3079038227604418


In [60]:
y_pred = model.predict(X_test)
r2_score(y_test , y_pred)

0.5736570855516909

# Manual implementation of Stochastic Gradient Descent Regression 

In [61]:
class ManualStochasticGDR:

    def __init__(self,learning_Rate=0.001,epochs=1000):
        self.learning_Rate = learning_Rate
        self.epochs = epochs
        self.intercept_ = None
        self.coef_ = None

    
    def fit(self , X_train , y_train):

        X_train = X_train.to_numpy()
        y_train = y_train.to_numpy()
    
        self.intercept_ = 0
        self.coef_ = np.zeros(X_train.shape[1])
        
        for i in range(self.epochs):
            for j in range(X_train.shape[0]):
                idx = np.random.randint(0 , X_train.shape[0])
                
                y_hat =  np.dot(X_train[idx] , self.coef_) + self.intercept_
                interceptDer = -2 * (y_train[idx] - y_hat)
                self.intercept_ = self.intercept_ - (self.learning_Rate * interceptDer)
        
                
                coefDer = -2 * (y_train[idx] - y_hat) * X_train[idx]
                self.coef_ = self.coef_ - (self.learning_Rate * coefDer)
        
    def predict(self , X_test):
        X_test = np.asarray(X_test)
        return np.dot(X_test , self.coef_) + self.intercept_

In [62]:
m = ManualStochasticGDR()

In [63]:
m.fit(X_train , y_train)

In [64]:
print(m.coef_)
print(m.intercept_)

[-4.27426551e-01  1.86012837e+00  5.41835029e-02 -1.05637071e-03
  7.62657600e-02 -2.81049783e-01  2.52378393e-01  6.22463856e-01
 -4.94393286e-01 -4.52579132e-01  0.00000000e+00 -7.38156214e-01
  0.00000000e+00]
-1.6851286314462015


In [65]:
y_pred = m.predict(X_test)
r2_score(y_test , y_pred)

0.5890708842071961

# Manual implementation of Mini-Batch Gradient Descent Regression 

In [94]:
class ManualMiniBatchGDR:

    def __init__(self,batch_size,learning_rate=0.01,epochs=100):
        
        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

    
    def fit(self , X_train , y_train):

        X_train = X_train.to_numpy()
        y_train = y_train.to_numpy()
    
        self.intercept_ = 0
        self.coef_ = np.zeros(X_train.shape[1])
        
        for i in range(self.epochs):
            for j in range(int(X_train.shape[0]/self.batch_size)):
                idx = random.sample(range(X_train.shape[0]),self.batch_size)
                
                y_hat = np.dot(X_train[idx],self.coef_) + self.intercept_
                #print("Shape of y_hat",y_hat.shape)
                intercept_der = -2 * np.mean(y_train[idx] - y_hat)
                self.intercept_ = self.intercept_ - (self.lr * intercept_der)

                coef_der = -2 * np.dot((y_train[idx] - y_hat),X_train[idx])
                self.coef_ = self.coef_ - (self.lr * coef_der)
        
    def predict(self , X_test):
        X_test = np.asarray(X_test)
        return np.dot(X_test , self.coef_) + self.intercept_

In [95]:
mbGDR = ManualStochasticGDR()

In [96]:
mbGDR.fit(X_train , y_train)

In [97]:
print(mbGDR.coef_)
print(mbGDR.intercept_)

[-0.44154052  1.85967399  0.05796706 -0.00985264  0.11229115 -0.25877833
  0.28066109  0.63119143 -0.50119996 -0.48897842  0.         -0.72142914
  0.        ]
[-1.71160752]


In [98]:
y_pred = m.predict(X_test)
r2_score(y_test , y_pred)

0.5890708842071961